<a href="https://colab.research.google.com/github/saniluttu/student-wellbeing-statistics-day16/blob/main/Student_Wellbeing_Statistics_Day16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Student Wellbeing — Statistical Analysis & Probability
**Day 16 Assignment**

This notebook analyzes the Student Wellbeing Survey dataset using descriptive statistics, outlier detection, probability theory (including conditional probability, independence, and Bayes' theorem), and the normal distribution. Each section shows the relevant formula, the Python calculation, the numerical answer, and a brief interpretation.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

pd.set_option('display.max_columns', None)
print("Pandas version:", pd.__version__)

Pandas version: 3.0.2


## 1. Loading and Inspecting the Dataset

In [ ]:
df = pd.read_csv('Day16_Student_Wellbeing_Survey.csv')
print("Shape:", df.shape)
df.head()

Shape: (600, 20)


,Student_ID,Age,Faculty,Year_of_Study,City,Accommodation,Scholarship,Part_Time_Job,Internet_Quality,Preferred_Study_Space,Weekly_Study_Hours,Average_Sleep_Hours,Daily_Screen_Time_Hours,Exercise_Days_Per_Week,Commute_Time_Minutes,Stress_Score,Academic_Readiness_Score,Overall_Satisfaction,Monthly_Discretionary_Spending,Social_Activity_Hours_Per_Week
0,STU0001,21,Data Science,3,Bengaluru,Hostel,No,Yes,Good,Café,15.2,5.7,3.2,3,11.8,5.4,63.1,3.2,5131.0,7.8
1,STU0002,21,Data Science,4,Pune,Hostel,Yes,No,Poor,Library,17.6,7.7,3.5,4,20.3,2.1,74.2,3.8,3508.0,9.2
2,STU0003,22,Life Sciences,2,Chandigarh,Home,Yes,No,Poor,Home,18.9,6.4,2.8,1,15.0,5.2,77.3,4.0,4584.0,7.7
3,STU0004,18,Business,4,Bengaluru,Hostel,Yes,Yes,Average,Café,19.5,6.7,3.0,5,12.1,4.7,67.0,4.0,5456.0,5.3
4,STU0005,20,Business,3,Jammu,Shared Apartment,Yes,No,Good,Home,18.4,7.3,3.0,1,30.2,5.4,65.6,3.2,11732.0,15.9


In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 20 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Student_ID                      600 non-null    str    
 1   Age                             600 non-null    int64  
 2   Faculty                         600 non-null    str    
 3   Year_of_Study                   600 non-null    int64  
 4   City                            600 non-null    str    
 5   Accommodation                   600 non-null    str    
 6   Scholarship                     600 non-null    str    
 7   Part_Time_Job                   600 non-null    str    
 8   Internet_Quality                600 non-null    str    
 9   Preferred_Study_Space           600 non-null    str    
 10  Weekly_Study_Hours              600 non-null    float64
 11  Average_Sleep_Hours             600 non-null    float64
 12  Daily_Screen_Time_Hours         600 non-null   

In [ ]:
df.isna().sum().sum()  # confirming no missing values

np.int64(0)

## 2. Central Tendency — Mean, Median, Mode

**Formulas:**
- Mean: $\bar{x} = \frac{\sum x_i}{n}$
- Median: the middle value of the sorted data (or average of the two middle values)
- Mode: the most frequently occurring value

In [ ]:
variables = ['Weekly_Study_Hours', 'Average_Sleep_Hours', 'Daily_Screen_Time_Hours',
             'Stress_Score', 'Academic_Readiness_Score']

central_tendency = pd.DataFrame({
    'Mean': df[variables].mean(),
    'Median': df[variables].median(),
    'Mode': df[variables].mode().iloc[0]
})
central_tendency.round(2)

,Mean,Median,Mode
Weekly_Study_Hours,15.69,15.40,13.1
Average_Sleep_Hours,7.00,7.00,7.0
Daily_Screen_Time_Hours,4.50,4.20,3.5
Stress_Score,4.46,4.50,4.7
Academic_Readiness_Score,71.77,71.65,71.1


**Interpretation:** For all five variables, the mean and median are close to each other, suggesting roughly symmetric distributions without extreme skew. The mode shows the single most common value, which is less useful here since these are continuous measurements with many near-unique values, but it still gives a sense of where data clusters most densely.

## 3. Measures of Spread — Range, Variance, Std Dev, Q1, Q3, IQR

**Formulas:**
- Range: $\text{max} - \text{min}$
- Variance: $s^2 = \frac{\sum (x_i - \bar{x})^2}{n-1}$
- Standard Deviation: $s = \sqrt{s^2}$
- IQR: $Q3 - Q1$

In [ ]:
spread_stats = pd.DataFrame({
    'Range': df[variables].max() - df[variables].min(),
    'Variance': df[variables].var(),
    'Std_Dev': df[variables].std(),
    'Q1': df[variables].quantile(0.25),
    'Q3': df[variables].quantile(0.75),
    'IQR': df[variables].quantile(0.75) - df[variables].quantile(0.25)
})
spread_stats.round(2)

,Range,Variance,Std_Dev,Q1,Q3,IQR
Weekly_Study_Hours,30.0,19.22,4.38,13.00,18.42,5.42
Average_Sleep_Hours,5.0,0.70,0.84,6.48,7.60,1.12
Daily_Screen_Time_Hours,11.2,3.47,1.86,3.20,5.40,2.20
Stress_Score,8.4,2.78,1.67,3.30,5.70,2.40
Academic_Readiness_Score,56.1,93.98,9.69,65.20,78.03,12.83


In [ ]:
most_variable = spread_stats['Std_Dev'].idxmax()
print(f"Variable with the greatest variability (highest standard deviation): {most_variable}")
print(f"Standard deviation: {spread_stats.loc[most_variable, 'Std_Dev']:.2f}")

Variable with the greatest variability (highest standard deviation): Academic_Readiness_Score
Standard deviation: 9.69


**Interpretation:** The variable with the greatest variability is the one with the highest standard deviation relative to its own scale — printed above. This tells us which aspect of student life varies the most across the surveyed population.

## 4. Outlier Detection Using the IQR Method

**Formula:** A value is an outlier if it falls below $Q1 - 1.5 \times IQR$ or above $Q3 + 1.5 \times IQR$.

In [ ]:
outlier_vars = ['Weekly_Study_Hours', 'Daily_Screen_Time_Hours',
                'Commute_Time_Minutes', 'Monthly_Discretionary_Spending']

def get_outlier_bounds(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return lower, upper

outlier_summary = {}
for col in outlier_vars:
    lower, upper = get_outlier_bounds(df, col)
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    outlier_summary[col] = len(outliers)
    print(f"{col}: {len(outliers)} outliers | bounds = [{lower:.2f}, {upper:.2f}]")

Weekly_Study_Hours: 8 outliers | bounds = [4.86, 26.56]
Daily_Screen_Time_Hours: 18 outliers | bounds = [-0.10, 8.70]
Commute_Time_Minutes: 4 outliers | bounds = [-13.85, 58.75]
Monthly_Discretionary_Spending: 20 outliers | bounds = [-1430.25, 13351.75]


## 5. Comparing Mean and Median Before and After Removing Outliers

Using **Monthly_Discretionary_Spending** (the variable with the most outliers) as the example.

In [ ]:
target_col = max(outlier_summary, key=outlier_summary.get)
print(f"Variable chosen for before/after comparison: {target_col} ({outlier_summary[target_col]} outliers)")

lower, upper = get_outlier_bounds(df, target_col)
before_mean = df[target_col].mean()
before_median = df[target_col].median()

df_no_outliers = df[(df[target_col] >= lower) & (df[target_col] <= upper)]
after_mean = df_no_outliers[target_col].mean()
after_median = df_no_outliers[target_col].median()

comparison = pd.DataFrame({
    'Before': [before_mean, before_median],
    'After': [after_mean, after_median]
}, index=['Mean', 'Median'])
comparison.round(2)

Variable chosen for before/after comparison: Monthly_Discretionary_Spending (20 outliers)


,Before,After
Mean,6343.78,5973.72
Median,5685.50,5590.50


**Interpretation:** The mean typically shifts more than the median after removing outliers, since the mean is sensitive to extreme values while the median is robust to them. A larger gap between the before/after mean than the before/after median confirms this expected behavior.

## 6. Probability — Defining the Events

- **Event A**: `Part_Time_Job` = "Yes"
- **Event B**: `Stress_Score` ≥ 7
- **Event C**: `Scholarship` = "Yes"
- **Event D**: `Exercise_Days_Per_Week` ≥ 3

In [ ]:
n = len(df)

A = df['Part_Time_Job'] == 'Yes'
B = df['Stress_Score'] >= 7
C = df['Scholarship'] == 'Yes'
D = df['Exercise_Days_Per_Week'] >= 3

print(f"Total students (n): {n}")
print(f"Event A count (Part-Time Job = Yes): {A.sum()}")
print(f"Event B count (Stress Score >= 7): {B.sum()}")
print(f"Event C count (Scholarship = Yes): {C.sum()}")
print(f"Event D count (Exercise >= 3 days/week): {D.sum()}")

Total students (n): 600
Event A count (Part-Time Job = Yes): 152
Event B count (Stress Score >= 7): 45
Event C count (Scholarship = Yes): 184
Event D count (Exercise >= 3 days/week): 356


### 6.1 Individual Probabilities

**Formula:** $P(X) = \dfrac{\text{count of } X}{n}$

In [ ]:
P_A = A.mean()
P_B = B.mean()
P_C = C.mean()
P_D = D.mean()

print(f"P(A) = {P_A:.4f}")
print(f"P(B) = {P_B:.4f}")
print(f"P(C) = {P_C:.4f}")
print(f"P(D) = {P_D:.4f}")

P(A) = 0.2533
P(B) = 0.0750
P(C) = 0.3067
P(D) = 0.5933


**Interpretation:** These probabilities represent the proportion of surveyed students who have a part-time job, report high stress, hold a scholarship, and exercise at least 3 days a week, respectively.

### 6.2 P(A or B) and P(A and B)

**Formulas:**
- $P(A \cup B) = P(A) + P(B) - P(A \cap B)$
- $P(A \cap B) = \dfrac{\text{count of } (A \text{ and } B)}{n}$

In [ ]:
P_A_and_B = (A & B).mean()
P_A_or_B = P_A + P_B - P_A_and_B

# Cross-check P(A or B) directly
P_A_or_B_direct = (A | B).mean()

print(f"P(A and B) = {P_A_and_B:.4f}")
print(f"P(A or B) [via formula] = {P_A_or_B:.4f}")
print(f"P(A or B) [direct count]  = {P_A_or_B_direct:.4f}")

P(A and B) = 0.0517
P(A or B) [via formula] = 0.2767
P(A or B) [direct count]  = 0.2767


**Interpretation:** P(A and B) is the chance a student both works part-time and reports high stress. P(A or B) is the chance a student is in at least one of those two groups — the formula-based and direct-count results match, confirming the calculation.

### 6.3 Conditional Probabilities P(A|B) and P(B|A)

**Formula:** $P(A|B) = \dfrac{P(A \cap B)}{P(B)}$, and symmetrically for $P(B|A)$.

In [ ]:
P_A_given_B = P_A_and_B / P_B
P_B_given_A = P_A_and_B / P_A

print(f"P(A|B) = {P_A_given_B:.4f}  (probability of having a part-time job, given high stress)")
print(f"P(B|A) = {P_B_given_A:.4f}  (probability of high stress, given a part-time job)")

P(A|B) = 0.6889  (probability of having a part-time job, given high stress)
P(B|A) = 0.2039  (probability of high stress, given a part-time job)


**Interpretation:** These two conditional probabilities answer different questions and are generally not equal — P(A|B) looks at part-time job rate among high-stress students, while P(B|A) looks at the high-stress rate among part-time working students.

## 7. Mutual Exclusivity — Year_of_Study = 1 vs. Year_of_Study = 4

**Definition:** Two events are mutually exclusive if they cannot both occur at the same time, i.e. $P(X \cap Y) = 0$.

In [ ]:
Year1 = df['Year_of_Study'] == 1
Year4 = df['Year_of_Study'] == 4

both = (Year1 & Year4).sum()
print(f"Students who are simultaneously Year 1 AND Year 4: {both}")
print(f"P(Year 1 and Year 4) = {(Year1 & Year4).mean():.4f}")

is_mutually_exclusive = both == 0
print(f"\nAre Year_of_Study = 1 and Year_of_Study = 4 mutually exclusive? {is_mutually_exclusive}")

Students who are simultaneously Year 1 AND Year 4: 0
P(Year 1 and Year 4) = 0.0000

Are Year_of_Study = 1 and Year_of_Study = 4 mutually exclusive? True


**Interpretation:** Since a student can only be in one year of study at a time, no student can simultaneously be Year 1 and Year 4 — the joint count is zero, confirming these events are **mutually exclusive**.

## 8. Checking Independence of A and B

**Definition:** Events A and B are independent if $P(A \cap B) = P(A) \times P(B)$.

In [ ]:
P_A_times_P_B = P_A * P_B

print(f"P(A and B)      = {P_A_and_B:.4f}")
print(f"P(A) x P(B)     = {P_A_times_P_B:.4f}")
print(f"Difference      = {abs(P_A_and_B - P_A_times_P_B):.4f}")

if abs(P_A_and_B - P_A_times_P_B) < 0.01:
    conclusion = "approximately independent (the values are very close)"
else:
    conclusion = "NOT independent (the values differ meaningfully)"

print(f"\nConclusion: Events A and B appear to be {conclusion}.")

P(A and B)      = 0.0517
P(A) x P(B)     = 0.0190
Difference      = 0.0327

Conclusion: Events A and B appear to be NOT independent (the values differ meaningfully).


**Interpretation:** If P(A and B) closely matches P(A) × P(B), having a part-time job and reporting high stress don't meaningfully influence each other's likelihood in this dataset. A large gap between the two values would instead suggest a real relationship — e.g. working part-time genuinely raising or lowering the chance of high stress.

## 9. Bayes' Theorem — Recomputing P(A|B)

**Formula:** $P(A|B) = \dfrac{P(B|A) \times P(A)}{P(B|A) \times P(A) + P(B|\text{not }A) \times P(\text{not }A)}$

In [ ]:
P_not_A = 1 - P_A
P_B_given_notA = (~A & B).sum() / (~A).sum()

P_A_given_B_bayes = (P_B_given_A * P_A) / (P_B_given_A * P_A + P_B_given_notA * P_not_A)

print(f"P(B|A)          = {P_B_given_A:.4f}")
print(f"P(B|not A)      = {P_B_given_notA:.4f}")
print(f"P(A)            = {P_A:.4f}")
print(f"P(not A)        = {P_not_A:.4f}")
print(f"\nP(A|B) via Bayes' theorem = {P_A_given_B_bayes:.4f}")
print(f"P(A|B) via direct calculation = {P_A_given_B:.4f}")
print(f"Match: {np.isclose(P_A_given_B_bayes, P_A_given_B)}")

P(B|A)          = 0.2039
P(B|not A)      = 0.0312
P(A)            = 0.2533
P(not A)        = 0.7467

P(A|B) via Bayes' theorem = 0.6889
P(A|B) via direct calculation = 0.6889
Match: True


**Interpretation:** Bayes' theorem lets us compute P(A|B) purely from P(B|A), P(B|not A), and P(A) — without directly counting the joint event. The result matches the direct conditional-probability calculation, confirming both approaches are consistent.

## 10. Normal Distribution — Academic_Readiness_Score

In [ ]:
ars_mean = df['Academic_Readiness_Score'].mean()
ars_std = df['Academic_Readiness_Score'].std()

print(f"Mean (Academic_Readiness_Score) = {ars_mean:.2f}")
print(f"Standard Deviation = {ars_std:.2f}")

Mean (Academic_Readiness_Score) = 71.77
Standard Deviation = 9.69


### 10.1 Z-scores for the Highest and Lowest Scores

**Formula:** $z = \dfrac{x - \mu}{\sigma}$

In [ ]:
max_score = df['Academic_Readiness_Score'].max()
min_score = df['Academic_Readiness_Score'].min()

z_max = (max_score - ars_mean) / ars_std
z_min = (min_score - ars_mean) / ars_std

print(f"Highest score = {max_score:.2f}  ->  Z-score = {z_max:.2f}")
print(f"Lowest score  = {min_score:.2f}  ->  Z-score = {z_min:.2f}")

Highest score = 98.00  ->  Z-score = 2.71
Lowest score  = 41.90  ->  Z-score = -3.08


**Interpretation:** The Z-score tells us how many standard deviations a value is from the mean. A positive Z-score (highest scorer) means that student is above average academic readiness; a negative Z-score (lowest scorer) means that student is below average. The magnitude shows how extreme each value is relative to the rest of the class.

### 10.2 The 68-95-99.7 Empirical Rule

**Rule:** For a normal distribution, approximately 68% of values fall within 1 standard deviation of the mean, 95% within 2, and 99.7% within 3.

In [ ]:
within_1sd = df[(df['Academic_Readiness_Score'] >= ars_mean - ars_std) &
                (df['Academic_Readiness_Score'] <= ars_mean + ars_std)].shape[0] / n * 100

within_2sd = df[(df['Academic_Readiness_Score'] >= ars_mean - 2*ars_std) &
                (df['Academic_Readiness_Score'] <= ars_mean + 2*ars_std)].shape[0] / n * 100

within_3sd = df[(df['Academic_Readiness_Score'] >= ars_mean - 3*ars_std) &
                (df['Academic_Readiness_Score'] <= ars_mean + 3*ars_std)].shape[0] / n * 100

empirical_rule = pd.DataFrame({
    'Expected_%': [68, 95, 99.7],
    'Actual_%': [round(within_1sd, 1), round(within_2sd, 1), round(within_3sd, 1)]
}, index=['Within 1 SD', 'Within 2 SD', 'Within 3 SD'])
empirical_rule

,Expected_%,Actual_%
Within 1 SD,68.0,69.5
Within 2 SD,95.0,95.2
Within 3 SD,99.7,99.8


**Interpretation:** Comparing the actual percentages of students falling within 1, 2, and 3 standard deviations of the mean to the theoretical 68-95-99.7 rule tells us how closely `Academic_Readiness_Score` follows a normal distribution. Close agreement supports treating this variable as approximately normally distributed.

## 11. Meaningful Statistical Observations

1. **Central tendency measures are closely aligned** across the five key variables (mean ≈ median for most), suggesting roughly symmetric distributions without severe skew in student study habits, sleep, screen time, stress, and academic readiness.
2. **Variability differs substantially by variable** — the variable with the highest standard deviation (identified above) reflects the aspect of student life with the widest spread of experiences across the surveyed population.
3. **Outliers are concentrated in spending and time-use variables** (e.g. Monthly_Discretionary_Spending), and removing them shifts the mean more than the median — a textbook illustration of the mean's sensitivity to extreme values versus the median's robustness.
4. **Part-time work and high stress show a measurable but not overwhelming relationship** — the gap (or lack thereof) between P(A and B) and P(A)×P(B) indicates whether working part-time meaningfully shifts stress risk, rather than being purely coincidental overlap.
5. **Academic_Readiness_Score behaves close to a normal distribution**, evidenced by the empirical rule check — the share of students within 1, 2, and 3 standard deviations of the mean tracks reasonably closely with the theoretical 68-95-99.7 benchmarks, supporting the use of Z-scores to meaningfully interpret individual student standing.